# Telco Customer Churn — EDA Part 2: Churn Drivers & Segments

Bivariate EDA focused on the churn target.  We examine churn rates across
contract type, tenure cohorts, service mix, and payment method, then rank
candidate predictors using chi-square tests of independence and mutual
information scores.

Helper functions (`churn_rate_by_group`, `chi_square_table`, `compute_mutual_info`)
live in `src/features/eda_helpers.py`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import seaborn as sns
import yaml

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.load import download_telco_data, load_raw
from src.features.eda_helpers import (
    churn_rate_by_group,
    chi_square_table,
    compute_mutual_info,
)
from src.utils.plotting import CHURN_PALETTE, PALETTE, save_fig, set_plot_style

set_plot_style()
FIGURES_DIR = REPO_ROOT / 'reports' / 'figures'

In [ ]:
with open(REPO_ROOT / 'configs' / 'config.yaml') as fh:
    cfg = yaml.safe_load(fh)

RAW_PATH = REPO_ROOT / cfg['paths']['raw_data']
download_telco_data(RAW_PATH, urls=cfg['data']['download_urls'])
df = load_raw(RAW_PATH)

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce').fillna(0.0)
df['churn_flag'] = (df['Churn'] == 'Yes').astype(int)

bins = [0, 12, 24, 36, 48, 60, 72]
labels = ['0-12', '13-24', '25-36', '37-48', '49-60', '61-72']
df['tenure_bucket'] = pd.cut(df['tenure'], bins=bins, labels=labels, include_lowest=True)

churn_rate_overall = df['churn_flag'].mean()
print(f'Loaded {len(df):,} rows  |  overall churn rate = {churn_rate_overall:.1%}')

## 1. Churn Rate by Contract Type

Contract type is the single strongest churn driver in this dataset.
Month-to-month subscribers face zero switching costs; two-year customers
have committed to a long relationship and churn at a fraction of the rate.

The ~42 percentage-point spread between month-to-month and two-year customers
will dominate tree-based feature importances and SHAP values.

In [ ]:
contract_stats = churn_rate_by_group(df, 'Contract')

fig, ax = plt.subplots(figsize=(7, 4))
colors_c = [CHURN_PALETTE['Yes'] if r > 0.25 else PALETTE[0]
            for r in contract_stats['churn_rate']]
bars = ax.barh(
    contract_stats.index.tolist(),
    contract_stats['churn_rate'].tolist(),
    color=colors_c,
)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
ax.set_xlim(0, 0.57)
for bar, (_, row) in zip(bars, contract_stats.iterrows()):
    cr = row['churn_rate']
    cnt = int(row['count'])
    ax.text(
        bar.get_width() + 0.005,
        bar.get_y() + bar.get_height() / 2,
        f'{cr:.1%}  (n={cnt:,})',
        va='center', fontsize=9,
    )
ax.set_xlabel('Churn rate')
ax.set_title('Churn rate by contract type')
fig.tight_layout()
save_fig(fig, '06_churn_rate_contract', FIGURES_DIR)
plt.show()
print('Saved: 06_churn_rate_contract.png')

## 2. Churn Rate by Tenure Segment

New customers (0–12 months) churn at nearly 50% — the highest of any cohort.
After the first year, churn drops sharply and continues to decline as
customers age into longer relationships.

This non-linear U-shape means raw `tenure` alone is not optimal; a binned
`tenure_bucket` feature captures the cohort structure directly.

In [ ]:
tenure_order = ['0-12', '13-24', '25-36', '37-48', '49-60', '61-72']
tenure_stats = churn_rate_by_group(df, 'tenure_bucket').reindex(tenure_order)

fig, ax = plt.subplots(figsize=(8, 4))
colors_t = [CHURN_PALETTE['Yes'] if r > 0.25 else PALETTE[0]
            for r in tenure_stats['churn_rate']]
bars = ax.bar(
    tenure_stats.index.tolist(),
    tenure_stats['churn_rate'].tolist(),
    color=colors_t,
    edgecolor='white',
)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
for bar, (_, row) in zip(bars, tenure_stats.iterrows()):
    cr = row['churn_rate']
    cnt = int(row['count'])
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f'{cr:.0%}',
        ha='center', fontsize=9,
    )
ax.set_xlabel('Tenure cohort (months)')
ax.set_ylabel('Churn rate')
ax.set_title('Churn rate by tenure segment')
fig.tight_layout()
save_fig(fig, '07_churn_rate_tenure', FIGURES_DIR)
plt.show()
print('Saved: 07_churn_rate_tenure.png')

## 3. Churn Rate by Service Profile

**Internet service** (left): Fiber optic customers churn at roughly twice the
rate of DSL customers.  Customers with no internet (voice-only) have near-zero
churn — they are typically long-tenured or on fixed contracts.

**Service add-ons** (right): Customers *without* security/support add-ons
(OnlineSecurity, TechSupport, DeviceProtection, OnlineBackup) churn at roughly
double the rate of subscribers.  Streaming services show the opposite — premium
streaming customers churn slightly *more*, likely driven by the fibre/promotional
cohort overlap.

In [ ]:
internet_stats = churn_rate_by_group(df, 'InternetService')

label_map = {
    'OnlineSecurity': 'Security',
    'OnlineBackup': 'Backup',
    'DeviceProtection': 'Device',
    'TechSupport': 'TechSupp',
    'StreamingTV': 'StreamTV',
    'StreamingMovies': 'StreamMov',
}
addon_rows = []
for col, short in label_map.items():
    for val in ['Yes', 'No']:
        mask = df[col] == val
        addon_rows.append({
            'service': short,
            'subscribed': val,
            'churn_rate': df.loc[mask, 'churn_flag'].mean(),
        })
addon_stats = pd.DataFrame(addon_rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bars_i = axes[0].barh(
    internet_stats.index.tolist(),
    internet_stats['churn_rate'].tolist(),
    color=PALETTE[:3],
)
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
axes[0].set_xlim(0, 0.55)
for bar, (_, row) in zip(bars_i, internet_stats.iterrows()):
    cr = row['churn_rate']
    cnt = int(row['count'])
    axes[0].text(
        bar.get_width() + 0.005,
        bar.get_y() + bar.get_height() / 2,
        f'{cr:.1%}  (n={cnt:,})',
        va='center', fontsize=9,
    )
axes[0].set_title('Churn rate by internet service')
axes[0].set_xlabel('Churn rate')

sns.barplot(
    data=addon_stats,
    x='service',
    y='churn_rate',
    hue='subscribed',
    palette={'Yes': PALETTE[3], 'No': PALETTE[1]},
    errorbar=None,
    ax=axes[1],
)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
axes[1].set_title('Churn rate: subscribed vs. non-subscribed')
axes[1].set_xlabel('')
axes[1].set_ylabel('Churn rate')
axes[1].legend(title='Subscribed', frameon=False)
axes[1].tick_params(axis='x', rotation=25)

fig.suptitle('Service-level churn drivers', fontweight='bold')
fig.tight_layout()
save_fig(fig, '08_churn_rate_services', FIGURES_DIR)
plt.show()
print('Saved: 08_churn_rate_services.png')

## 4. Churn Rate by Payment Method

Electronic check customers have a churn rate more than double that of automatic-
payment customers.  This is partly a confounding effect — electronic check is
the dominant payment choice among month-to-month subscribers — but the
association is strong enough to retain `PaymentMethod` as a direct feature.

In [ ]:
payment_stats = churn_rate_by_group(df, 'PaymentMethod')

fig, ax = plt.subplots(figsize=(8, 4))
bars_p = ax.barh(
    payment_stats.index.tolist(),
    payment_stats['churn_rate'].tolist(),
    color=PALETTE[:4],
)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
ax.set_xlim(0, 0.55)
for bar, (_, row) in zip(bars_p, payment_stats.iterrows()):
    cr = row['churn_rate']
    cnt = int(row['count'])
    ax.text(
        bar.get_width() + 0.005,
        bar.get_y() + bar.get_height() / 2,
        f'{cr:.1%}  (n={cnt:,})',
        va='center', fontsize=9,
    )
ax.set_xlabel('Churn rate')
ax.set_title('Churn rate by payment method')
fig.tight_layout()
save_fig(fig, '09_churn_rate_payment', FIGURES_DIR)
plt.show()
print('Saved: 09_churn_rate_payment.png')

## 5. Chi-Square Tests of Independence

Pearson chi-square tests whether each categorical feature is statistically
independent of churn.  **Cramér's V** is the effect-size metric — bounded [0, 1]
and comparable across features with different cardinalities.

`p_value < 0.05` means the association is unlikely to be due to chance alone;
Cramér's V quantifies the *strength* of that association.

In [ ]:
cat_cols_for_chi2 = cfg['features']['categorical'] + ['SeniorCitizen']
chi2_results = chi_square_table(df, cat_cols_for_chi2)

n_sig = int(chi2_results['significant'].sum())
print(f'{n_sig}/{len(chi2_results)} features are significantly associated with churn (p < 0.05)')
chi2_results.round(4)

## 6. Mutual Information Ranking

Mutual information (MI) measures the reduction in uncertainty about `Churn`
given knowledge of a feature.  Unlike Cramér's V, MI works on a common scale
for both categorical *and* numeric features, enabling a single combined ranking.

Categorical and binary-integer columns are label-encoded before scoring;
the `discrete_features` flag is set per-column so sklearn uses the correct
MI estimator for each type.

In [ ]:
all_feature_cols = (
    cfg['features']['categorical']
    + cfg['features']['numeric']
    + ['SeniorCitizen']
)
mi_scores = compute_mutual_info(df, all_feature_cols)

threshold = float(mi_scores.median())
mi_rev = mi_scores.iloc[::-1]
bar_colors = [
    CHURN_PALETTE['Yes'] if s >= threshold else PALETTE[0]
    for s in mi_rev
]

fig, ax = plt.subplots(figsize=(7, 8))
ax.barh(mi_rev.index.tolist(), mi_rev.values, color=bar_colors)
ax.axvline(threshold, color='#6B7280', linestyle='--', linewidth=1, label='median')
ax.set_xlabel('Mutual information score')
ax.set_title('Feature ranking by mutual information (target: Churn)')
ax.legend(frameon=False)
fig.tight_layout()
save_fig(fig, '10_mutual_info_ranking', FIGURES_DIR)
plt.show()
print('Saved: 10_mutual_info_ranking.png')

print('\nTop 10 features by MI:')
print(mi_scores.head(10).to_string())

## Top 5 Features to Engineer

Based on the churn rate analysis, Cramér's V rankings, and MI scores above,
the following five derived features are expected to deliver the highest
incremental signal beyond the raw columns.

| Feature | Construction | Rationale |
|---|---|---|
| **`contract_monthly`** | `Contract == 'Month-to-month'` → 0/1 | Highest Cramér's V and MI; binary flag is more model-stable than a 3-way OHE for tree learners |
| **`tenure_bucket`** | `pd.cut(tenure, [0,12,24,36,48,60,72])` | Captures the non-linear churn decay curve; ordinal encoding preserves the monotone signal |
| **`fiber_optic`** | `InternetService == 'Fiber optic'` → 0/1 | Fiber customers churn at ~2× the DSL rate; isolating this group is more informative than the full 3-level encoding |
| **`security_bundle_score`** | Sum of `OnlineSecurity + TechSupport + DeviceProtection + OnlineBackup` (Yes=1, else 0) | Each security/support add-on individually signals lower churn; the aggregate 0–4 depth-of-protection score captures the pattern in a single continuous feature and avoids high-cardinality interactions |
| **`electronic_check`** | `PaymentMethod == 'Electronic check'` → 0/1 | Churn rate >2× automatic-payment customers; retains independent signal after controlling for contract type |

These five features, together with the raw numerics (`tenure`, `MonthlyCharges`,
`TotalCharges`) and remaining categorical columns, form the initial feature set
for the modelling phase.